In [1]:
# !pip install -U pip
# !pip install alibi
# !pip install transformers pandas seaborn matplotlib scipy scikit-learn statsmodels textblob nltk numpy
# !pip install torch torchvision torchaudio
# !pip install -U sentence-transformers

  Using cached pip-24.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 22.0.4
    Uninstalling pip-22.0.4:
      Successfully uninstalled pip-22.0.4


In [1]:
!pip install torch transformers numpy
!pip install 'alibi[torch]'

  Using cached torch-2.2.2-cp39-none-macosx_10_9_x86_64.whl (150.8 MB)
  Using cached transformers-4.42.4-py3-none-any.whl (9.3 MB)
  Using cached numpy-2.0.0-cp39-cp39-macosx_10_9_x86_64.whl (21.2 MB)
  Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
  Using cached jinja2-3.1.4-py3-none-any.whl (133 kB)
  Using cached sympy-1.13.0-py3-none-any.whl (6.2 MB)
  Using cached fsspec-2024.6.1-py3-none-any.whl (177 kB)
  Using cached filelock-3.15.4-py3-none-any.whl (16 kB)
  Using cached huggingface_hub-0.23.4-py3-none-any.whl (402 kB)
  Using cached regex-2024.5.15-cp39-cp39-macosx_10_9_x86_64.whl (281 kB)
  Using cached tokenizers-0.19.1-cp39-cp39-macosx_10_12_x86_64.whl (2.5 MB)
  Using cached tqdm-4.66.4-py3-none-any.whl (78 kB)
  Using cached numpy-1.26.4-cp39-cp39-macosx_10_9_x86_64.whl (20.6 MB)
  Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Using cached safetensors-0.4.3-cp39-cp39-macosx_10_12_x86_64.whl (416 kB)
  Using cached PyYAML-6.0.1-cp39-cp39-macosx_10_9_x86

In [4]:
import torch

In [1]:
from transformers import RobertaForSequenceClassification

/Users/alexanders/ml-project-2-peaceduke-1/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import RobertaTokenizer

In [3]:
from alibi.explainers import IntegratedGradients

In [5]:
# Load pre-trained model and tokenizer
model_name = 'cardiffnlp/twitter-roberta-base-sentiment'
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name)

# Define a function to calculate sentiment scores and integrated gradients
def compute_sentiments(sentence):
    # Tokenize the input sentence
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
        scores = outputs.logits.softmax(dim=-1).squeeze()
    
    # Set up Integrated Gradients
    def predict(inputs):
        with torch.no_grad():
            logits = model(**inputs).logits
        return logits

    ig = IntegratedGradients(predict, layer=model.roberta.embeddings)
    attributions = ig.attribute(inputs.input_ids, target=1, n_steps=50)
    
    # Decode tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs.input_ids.squeeze().tolist())
    
    return tokens, scores, attributions.sum(dim=2).squeeze().tolist()

# Example sentence
sentence = "I love you, but I also kind of dislike you"
tokens, scores, attributions = compute_sentiments(sentence)

# Print results
print("Tokens:", tokens)
print("Scores:", scores)
print("Attributions:", attributions)

# Optional: Visualize the attributions
import matplotlib.pyplot as plt

def visualize_attributions(tokens, attributions):
    plt.figure(figsize=(10, 2))
    colors = ['red' if attr < 0 else 'green' for attr in attributions]
    plt.bar(range(len(tokens)), attributions, color=colors)
    plt.xticks(range(len(tokens)), tokens, rotation='vertical')
    plt.xlabel('Tokens')
    plt.ylabel('Attribution Values')
    plt.title('Integrated Gradients Attributions')
    plt.show()

visualize_attributions(tokens, attributions)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ImportError: Attempted to use IntegratedGradients without the correct optional dependencies installed. This may be due to missing or incompatible versions of dependencies. To install the correct optional dependencies, run `pip install alibi[tensorflow]` from the command line. For more information, check the installationdocumentation at https://docs.seldon.io/projects/alibi/en/latest/overview/getting_started.html.